In [ ]:
# Se non lo si è già fatto, installare la libreria fastf1 
!pip install fastf1

In [3]:
# Importo le librerie necessarie all'esecuzione del codice
import numpy as np
import pandas as pd
import fastf1
import datetime as dt

## Eventi

In [ ]:
# Creo le liste che conterranno i dati degli eventi
EventKey =[]
Country = []
EventName = []

# Ciclo for per estrarre i dati da tutti gli eventi del 2025
for i in range (1, 25):
    evento = fastf1.get_event(2025, i)
    EventKey.append(evento.RoundNumber)
    Country.append(evento.Country)
    EventName.append(evento.EventName)

# Creo il DataFrame dalle liste e lo salvo in csv
Eventi = pd.DataFrame({
    "EventKey": EventKey,
    "Country": Country,
    "EventName": EventName
})

Eventi.to_csv(r'Eventi.csv', index=False)


## Gomme

In [ ]:
# Creo manualmente il dizionario delle Gomme usate in F1 e lo salvo in csv

Gomme = {1:'C1', 2:'C2', 3:'C3', 4:'C4', 5:'C5', 6:'C6', 7:'INT', 8:'WET' }

Gomme_df = pd.DataFrame.from_dict(Gomme, orient=  'index', columns = ['Tyre'])

Gomme_df.to_csv('Gomme.csv', index_label='TyreKey')

## Piloti

In [ ]:
# Creo la lista che conterra i df di ogni evento
Events = []

#Ciclo for per estrarre i dati di tutti i piloti che hanno partecipato ad almeno una gara nel 2025
for i in range(1, 17):
    Event = fastf1.get_session(2025, i, 'R')
    Event.load()
    Drivers_Event = Event.results

    # Riorganizzo le colonne del df e lo aggiungo alla lista
    Drivers_Event = Drivers_Event.loc[:, ['DriverNumber', 'Abbreviation', 'TeamName', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl']]
    Events.append(Drivers_Event)

# Concateno tutti i df in un unico df, rimuovo i duplicati e creo la colonna DriverKey
Drivers = pd.concat(Events, ignore_index=True)
Drivers = Drivers.drop_duplicates(subset=['DriverNumber', 'TeamName'])
Drivers['DriverKey'] = range(1, len(Drivers) + 1)

# Riorganizzo le colonne e salvo in csv
Drivers = Drivers.loc[:,['DriverKey', 'DriverNumber', 'Abbreviation', 'TeamName', 'FirstName',                  
       'LastName', 'FullName', 'HeadshotUrl']]

Drivers.to_csv(r'Piloti_2025.csv', index=False)

## Team

In [ ]:
# Estraggo i nomi dei Team del 2025, aggiungo la colonna TeamKey e salvo in csv
team = fastf1.get_session(2025, 'Australia', 'Q')
team.load()
team =team.results
team = team.loc[:,['TeamName']].drop_duplicates()
team = team.reset_index(drop=True)
team['TeamKey'] = range(1, len(team) + 1)
team = team[['TeamKey', 'TeamName']]
team.to_csv('Team_2025.csv', index=False)

## Race Laps

In [ ]:
# Creo la lista che conterra i df di ogni gara
race_list = []

# Ciclo for per estrarre i dati di ogni gara disputata nel 2025 (max 25 nel range, le gare sono 24)
for i in range(1, 19):
    race = fastf1.get_session(2025, i, 'R')
    race.load()
    race = race.laps

    # Aggiungo la colonna EventKey e aggiungo il df alla lista
    race['EventKey'] = i
    race_list.append(race)

# Creo il dizionario delle gomme usate in ogni gara e lo converto in df
Tyre = {'EventKey': [i for i in range(1, 19) for _ in range(5)], 
        'Compound':['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET']*18 , 
        'TyreKey':[5, 4, 3, 7, 8, 4, 3, 2, 7, 8, 3, 2, 1, 7, 8, 3, 2, 1, 7, 8, 5, 4, 3, 7, 8, 5, 4, 3, 7, 8, 6, 5, 4, 7, 8, 6, 5, 4, 7, 8, 3, 2, 1, 7, 8, 6, 5, 4, 7, 8,
                    5, 4, 3, 7, 8, 4, 3, 2, 7, 8, 4, 3, 1, 7, 8, 5, 4, 3, 7, 8, 4, 3, 2, 7, 8, 5, 4, 3, 7, 8, 6, 5, 4, 7, 8, 5, 4, 3, 7, 8]
    }

Tyre_df = pd.DataFrame(Tyre)    

# Concateno tutti i df delle gare in un unico df
race_laps = pd.concat(race_list, ignore_index=True)

# Aggiungo la colonna TyreKey tramite merge
race_laps = pd.merge(race_laps, Tyre_df, on=['EventKey', 'Compound'], how='left')


# Recupero il df piloti e aggiungo la colonna DriverKey tramite merge
piloti = pd.read_csv(r'Piloti_2025.csv')

# Sistemo i tipi di dato della colonna per rendere possibile il merge
race_laps['DriverNumber'] = race_laps['DriverNumber'].astype('int64')
race_laps = pd.merge(race_laps, piloti[['DriverNumber', 'DriverKey']], on = 'DriverNumber', how = 'left')

# Sistemo i casi in cui i piloti hanno cambiato team durante la stagione
mask = race_laps["DriverNumber"] == 30
race_laps.loc[mask, "DriverKey"] = np.where(
    race_laps.loc[mask, "EventKey"] > 2,
    22,
    15
)

mask = race_laps["DriverNumber"] == 22
race_laps.loc[mask, "DriverKey"] = np.where(
    race_laps.loc[mask, "EventKey"] > 2,
    21,
    12
)

# Riorganizzo le colonne e sistemo i tipi di dato delle colonne temporali
race_laps = race_laps.loc[:,['DriverKey', 'EventKey', 'LapNumber', 'Position', 'LapTime',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'TyreKey', 'Compound',
       'Stint', 'TyreLife', 'TrackStatus']]


# Converto le colonne temporali in secondi totali
race_laps[['LapTime', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']] = race_laps[['LapTime', 'PitOutTime',
                                                                                                             'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']].apply(pd.to_timedelta)

race_laps[['LapTime', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']] = race_laps[['LapTime', 'PitOutTime', 'PitInTime', 'Sector1Time', 
                                                                                                            'Sector2Time', 'Sector3Time']].apply(lambda x: pd.to_timedelta(x).dt.total_seconds())

# Salvo in csv
race_laps.to_csv(r'Race_laps.csv', index=False)

## Risultati

In [ ]:
# Creo la lista che conterra i df di ogni evento
event_list = []

# Ciclo for per estrarre i dati di ogni evento disputato nel 2025 (max 25 nel range, le gare sono 24)
for i in range (1, 19):

    # Estraggo i dati delle qualifiche
    Quali = fastf1.get_session(2025, i, 'Q')
    Quali.load()
    risultati_Q =Quali.results 
    # Seleziono le colonne che mi interessano
    risultati_Q = risultati_Q.loc[:,['DriverNumber', 'Position','Q1','Q2','Q3']]

    # Estraggo i dati delle gare
    Gara= fastf1.get_session(2025, i, 'R')
    Gara.load()
    risultati_R = Gara.results
    #Seleziono le colonne che mi interessano
    risultati_R = risultati_R.loc[:,['DriverNumber', 'Position', 'Time', 'Status', 'Points']]

    # Unisco i due df
    risultati_evento = pd.merge(risultati_Q, risultati_R, on='DriverNumber', how='outer')

    #Aggiungo la colonna EventKey, rinomino le colonne e sistemo i tipi di dato delle colonne temporali
    risultati_evento['EventKey']= i
    risultati_evento.columns = ['DriverNumber', 'GridPosition', 'Q1', 'Q2', 'Q3', 'RacePosition', 'RaceTime', 'Status', 'Points', 'EventKey']
    risultati_evento['Q1'] = risultati_evento['Q1'].dt.total_seconds()
    risultati_evento['Q2'] = risultati_evento['Q2'].dt.total_seconds()
    risultati_evento['Q3'] = risultati_evento['Q3'].dt.total_seconds()
    risultati_evento['RaceTime'] = risultati_evento['RaceTime'].dt.total_seconds()

    # Ordino i dati in base alla posizione finale in gara
    risultati_evento = risultati_evento.sort_values(by='RacePosition')

    # Creo una colonna con il tempo in qualifica del pilota, per farlo devo unire le colonne Q3, Q2 e Q1

    # Prendo per ogni riga il miglior tempo disponibile tra Q3, Q2, Q1
    risultati_evento['QualiTime'] = risultati_evento[['Q3', 'Q2', 'Q1']].bfill(axis=1).iloc[:, 0]

    # Creo una colonna con la differenza tra il tempo segnato dal pilota e il tempo della pole position

    # Trova il miglior tempo assoluto in Q3 (escludendo NaN)
    miglior_Q3 = risultati_evento['Q3'].min(skipna=True)

    # Creo la colonna contenente la differenza rispetto al miglior Q3
    risultati_evento['QualiDiff'] = risultati_evento['QualiTime'] - miglior_Q3

    # Aggiungo il df dell'evento alla lista
    event_list.append(risultati_evento)

# Concateno tutti i df degli eventi in un unico df
risultati = pd.concat(event_list, ignore_index=True)

# Sistemo il tipo di dato della colonna DriverNumber per permettere il merge
risultati['DriverNumber'] = risultati['DriverNumber'].astype('int64')

# Recupero il df piloti e aggiungo la colonna DriverKey tramite merge
piloti = pd.read_csv(r'Piloti_2025.csv')
ris = pd.merge(risultati, piloti[['DriverNumber', 'DriverKey']], on = 'DriverNumber', how = 'left')

# Sistemo i casi in cui i piloti hanno cambiato team durante la stagione
mask1 = ris["DriverNumber"] == 30
ris.loc[mask1, "DriverKey"] = np.where(
    ris.loc[mask1, "EventKey"] > 2,
    22,
    15
)

mask2 = ris["DriverNumber"] == 22
ris.loc[mask2, "DriverKey"] = np.where(
    ris.loc[mask2, "EventKey"] > 2,
    21,
    12
)

# Elimino i duplicaro e riorganizzo le colonne e seleziono quelle che mi interessano
ris = ris.drop_duplicates()
ris = ris.loc[:,['DriverKey', 'EventKey', 'RacePosition', 'Points', 'RaceTime', 'Status',
       'GridPosition', 'Q1', 'Q2', 'Q3', 'QualiTime', 'QualiDiff',
       ]]

# Salvo in csv
ris.to_csv(r'risultati_2025.csv')

## Telemetrie

In [ ]:
# Creo la lista che conterra i df di ogni pilota in ogni evento
telemetrie = []

# Ciclo for per estrarre i dati di ogni evento disputato nel 2025 (max 25 nel range, le gare sono 24)
for i in range (1, 19):

    # Estraggo i dati di tutti i giri delle qualifiche
    qualifiche = fastf1.get_session(2025, i, 'Q')
    qualifiche.load()
    qualifiche = qualifiche.laps
    # Ciclo for per estrarre i dati di telemetria del miglior giro di ogni pilota che ha preso parte alle qualifiche escludendo i piloti senza dati di telemetria
    for driver in qualifiche['Driver'].unique():
        try:

            driver_laps = qualifiche.pick_drivers(driver)
            driver_telemetry = driver_laps.pick_fastest().get_telemetry()
            driver_telemetry['Driver'] = driver

            #Aggiungo la colonna EventKey, converto la colonna Time in secondi totali e seleziono le colonne che mi interessano
            driver_telemetry['EventKey']= i  
            driver_telemetry['Distance'] = driver_telemetry['Distance'].round(0)
            driver_telemetry['Time'] =driver_telemetry['Time'].dt.total_seconds()
            driver_telemetry = driver_telemetry.loc[:,['Driver', 'EventKey', 'Time','RPM', 'Speed', 'nGear', 'Throttle', 
                                                       'Brake', 'DRS', 'Distance', 'RelativeDistance', 'X', 'Y', 'Z']]
            
            # Aggiungo il df del pilota alla lista
            telemetrie.append(driver_telemetry)
        except:
            print(f"Driver {driver} has no telemetry data.")

# Concateno tutti i df delle gare in un unico df
telemetria = pd.concat(telemetrie, ignore_index=True)

# Recupero il df piloti e aggiungo la colonna DriverKey tramite merge
piloti = pd.read_csv(r'Piloti_2025.csv')
tel = pd.merge(telemetria, piloti[['Abbreviation', 'DriverKey']], left_on= 'Driver', right_on= 'Abbreviation', how = 'left')

# Sistemo i casi in cui i piloti hanno cambiato team durante la stagione
mask1 = tel["Driver"] == 'LAW'
tel.loc[mask1, "DriverKey"] = np.where(
    tel.loc[mask1, "EventKey"] > 2,
    22,
    15
)

mask2 = tel["Driver"] == 'TSU'
tel.loc[mask2, "DriverKey"] = np.where(
    tel.loc[mask2, "EventKey"] > 2,
    21,
    12
)

# Elimino i duplicaro,  riorganizzo le colonne e seleziono quelle che mi interessano, salvo in csv
tel = tel.drop_duplicates()
tel = tel.loc[:,['DriverKey', 'EventKey', 'Time',
            'RPM', 'Speed', 'nGear', 'Throttle', 'Brake', 'DRS', 
            'Distance', 'RelativeDistance']]
tel.to_csv(r'telemetria_Q_2025.csv', index=False)
